In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Preferences: a reward from comparisons

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position: Part C (application), session 2 of 2**
    - the last session treated a language model as a policy and listed where its reward can come from: a program, a learned reward model, a judge
    - **the limitation we start from:** for open-ended tasks (style, helpfulness) we cannot write a verifier, and numeric scores are hard to give
    - this session introduces the data that fixes it: **preference data**
- **The question:** where does a reward come from when we can only compare two responses?
- **By the end you can**
    - say what preference data is and why comparisons are often easier than scores
    - state the Bradley-Terry model and what fitting a reward model means
    - place a learned reward in the Part C training loop (RLHF)
    - say where direct preference optimization (DPO) stands relative to the policy gradient of Part B
    - *RLHF: reinforcement learning from human feedback; DPO: direct preference optimization*


# Preferences versus rewards

**Why this section?** The reward of Parts A and B was a number from the environment. Some problems offer only a ranking.

- a **reward** is a number: any two outputs can be compared, and the size of the gap means something
- a **preference** is a comparison: "is output A better than output B?"
    - we usually observe only *some* pairs, and no magnitude
- for example: I may prefer chocolate to vanilla, but I cannot say how much more
- style is a preference problem
    - **prompt:** "How do I change a tire?"
    - reply A: an accurate step-by-step answer
    - reply B: a correct but terse answer that leaves out the details (jack placement, tightening order)
    - both are acceptable, but most people prefer A


## Preference data

**Why this section?** The unit of data for everything that follows.

- a **preference example** is a triple $(\x, \y^+, \y^-)$
    - the input (prompt) $\x$
    - the preferred output $\y^+$
    - the non-preferred output $\y^-$
- both outputs may be *acceptable*
- the data records a quality ordering, not correctness
- such data can be used in two ways: imitate the preferred response (supervised fine-tuning, SFT), or **learn a reward from the comparisons** and optimize it with reinforcement learning, where the reward model learns from the *contrast* between the preferred and the non-preferred response
- why comparisons can be easier to collect than scores: raters often disagree about the scale of a score (is this a 7 or an 8?) and tend to agree more often about which of two is better (agreement on pairs is high but far from perfect)
- a worked triple: *prompt* "How do I change a tire?"
- *preferred* the full step-by-step answer
- *non-preferred* the terse one


## The case for preferences

**Why this section?** When is a comparison a better signal than a score?

| Scenario | Why preference data? | Typical example |
|:--|:--|:--|
| aligning a language model | no checker exists; humans often rank more consistently than they score | helpfulness, tone |
| hard-to-define or subjective success | comparisons are often easier to judge than scores | dialogue, safety, style |
| raters who use scales differently | a comparison removes each rater's personal scale (it does not remove noise or bias) | creative writing, ratings from different people |

- the price: a comparison carries less information than a score (no magnitude), raters can still disagree, and the Bradley-Terry model (next section) assumes one shared latent score


## From comparisons to a reward: the Bradley-Terry model

**Why this section?** How a pile of comparisons becomes the reward model of the last session. (It is also the model that DPO later assumes.)

- model a comparison (the **Bradley-Terry model**) by the difference of two scores from a **reward model** $\rew_\phi(\x, \y)$ (a network with parameters $\phi$)
    - $P(\y^+ \succ \y^-) = \sigma\bigl(\rew_\phi(\x, \y^+) - \rew_\phi(\x, \y^-)\bigr)$, with $\sigma$ the sigmoid
- $\succ$ reads "is preferred to"
- $\sigma(z) = 1/(1 + e^{-z})$
- the reward model is typically a language model with a scalar output on top
- fit $\phi$ by maximizing the likelihood of the observed comparisons: choose scores that agree with the rankings, by minimizing
    - $-\log \sigma\bigl(\rew_\phi(\x, \y^+) - \rew_\phi(\x, \y^-)\bigr)$, averaged over the dataset
- by numbers: the model scores $\y^+$ at $2.0$ and $\y^-$ at $0.5$
    - the difference is $1.5$, so $P(\y^+ \succ \y^-) = \sigma(1.5) \approx 0.82$ and the loss is $-\log 0.82 \approx 0.20$
    - training pushes the preferred score up and the other down
    - the loss can always be lowered by widening the gap, so reward models are trained briefly or regularized, to avoid fitting noise

<img src="images/c2_bradley_terry.png" width=85%>

- only **differences** matter: adding a constant (which may depend on the prompt) to every score of that prompt changes nothing, which is why comparisons need no common scale
- it also means the reward has no absolute zero
- here $\rew_\phi(\x, \y)$ is a score for a *whole response*, not a per-step reward


## Putting the reward model in the loop (RLHF)

**Why this section?** This is the route from preferences back to the Part B machinery.

- the pipeline, in three steps
    - **supervised fine-tuning:** a model that already produces reasonable responses (session C1: RL amplifies what the model already does)
    - **reward model:** fit $\rew_\phi$ on preference data (the Bradley-Terry loss above)
    - **reinforcement learning:** run the Part C loop (sample, score with $\rew_\phi$, advantages, clipped step, KL penalty to the reference)
- nothing new in the optimizer: the reward model is just another source in the table of the last session
- the cost: extra models (the reward model and the reference, plus a critic if PPO is used), and a proxy that can be hacked (the last session)
- the KL penalty reduces how far the policy can drift to exploit it
- the reference model is typically the supervised fine-tuned model of step 1


## An example preference dataset

**Why this section?** A concrete case, so the triple is not abstract.

- [UltraFeedback (binarized)](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized): columns *prompt*, *chosen*, *rejected*
- it trains an assistant to be helpful (answers the prompt, does not evade) and honest (truthful)
- how it was built
    - prompts gathered from many sources
    - responses to each prompt generated by several different language models
    - a strong language model (GPT-4) acts as the annotator: it critiques each response on several aspects (instruction-following, truthfulness, honesty and helpfulness) and gives scores, which rank the responses
- the **binarized** version keeps one pair per prompt: the best-scored response as *chosen* (by the dataset's own rating; versions differ in how that rating is computed) and one of the others at random as *rejected*
- note: **AI feedback** (synthetic data), not human feedback
- a model judge has its own biases (for example toward length), which a reward model trained on this data can inherit


## Where DPO stands

**Why this section?** DPO uses the same preference data, but is not one of Part B's methods.

- **direct preference optimization** (DPO) skips the *explicit* reward model and the sampling loop
    - it trains the policy on the stored triples with a classification-style loss: $-\log \sigma\bigl(\beta [\, \log \tfrac{\pi_\theta(\y^+ \mid \x)}{\pi_{\text{ref}}(\y^+ \mid \x)} - \log \tfrac{\pi_\theta(\y^- \mid \x)}{\pi_{\text{ref}}(\y^- \mid \x)} \,]\bigr)$, with $\beta$ a scale
    - here $\pi_\theta$ is the policy being trained, $\pi_{\text{ref}}$ the frozen reference model, and $\beta$ a scale (the same kind of weight as the KL penalty of C1)
    - the preferred response should gain probability, relative to the reference, more than the non-preferred one
    - the term $\beta \log (\pi_\theta / \pi_{\text{ref}})$ plays the role of the reward, so the loss has the Bradley-Terry form of this session
    - it uses whole-response probabilities (the product of the token probabilities, from the aside of the last session)
- relation to Part B
    - it is **not a variant of the policy gradient theorem**: it is not derived from it, it uses no sampled episodes or returns, and its ratio is to the reference model, not to an old policy (its update does resemble a weighted raising and lowering of log-probabilities)
    - standard DPO is **offline** (fixed data)
    - the loop of Part C is **online** (fresh samples from the current policy)
    - in RLHF the pairs are typically sampled from the supervised fine-tuned model
    - in a dataset like UltraFeedback the responses come from other models, so the data are off-policy for the model being trained
    - it is derived from the same KL-regularized reward objective that RLHF optimizes: the policy's log-ratio to the reference plays the role of a reward, and the Bradley-Terry model of this session is then assumed
    - the two are close cousins, but different algorithms
- the details, the derivation, and when each works better are the subject of the Advanced course (the course after this series, on reinforcement fine-tuning of language models)
- check: what if the model scores the pair the other way, a gap of $-1.5$? $\sigma(-1.5) \approx 0.18$ and the loss is about $1.70$


# What we established, and what it sets up

- **Established**
    - preference data records comparisons $(\x, \y^+, \y^-)$, often easier to collect than scores
    - the Bradley-Terry model turns score differences into comparison probabilities
    - fitting it gives a **reward model**
    - RLHF is the Part C loop with the learned reward model as its reward and a KL penalty to a reference
    - standard DPO uses the same data directly and offline
    - it is not a variant of the policy gradient theorem
- **Sets up**
    - the recap: all the methods of the series on one grid
    - the Advanced course on reinforcement fine-tuning of language models
